In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='/Users/toszmac/Documents/Pharma_Manufacturing_Data_Analysis_Project/Pharma_Manufacturing_Data_Analysis_Folder/.env')

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to lli_db")

Connected to lli_db


## Q1 - What is the average lead time per dosage form?

In [2]:
lead_time_dosage_form_query = """
    WITH batch_dates AS (
        SELECT
            f.jo_id,
            p.dosage_form,
            MAX(CASE WHEN f.stage = 'dry_blending' THEN d.full_date END) AS compounding_date,
            MAX(CASE WHEN f.stage = 'compression' THEN d.full_date END) AS compression_date,
            MAX(CASE WHEN f.stage = 'encapsulation' THEN d.full_date END) AS encapsulation_date,
            MAX(CASE WHEN f.stage = 'coating' THEN d.full_date END) AS coating_date
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        LEFT JOIN dim_date d ON f.date_id = d.date_id
        GROUP BY f.jo_id, p.dosage_form
    ),
    batch_lead_time AS (
        SELECT
            jo_id,
            dosage_form,
            compounding_date,
            CASE
                WHEN dosage_form = 'CAPSULE'
                    THEN encapsulation_date
                WHEN dosage_form IN ('FILM-COATED TABLET', 'SUSTAINED-RELEASE TABLET',
                                     'EXTENDED RELEASE TABLET', 'ENTERIC-COATED TABLET',
                                     'MODIFIED RELEASE TABLET')
                    THEN coating_date
                WHEN dosage_form IN ('TABLET', 'BILAYER TABLET')
                    THEN compression_date
                ELSE GREATEST(
                    COALESCE(compression_date, compounding_date),
                    COALESCE(encapsulation_date, compounding_date),
                    COALESCE(coating_date, compounding_date)
                )
            END AS end_date
        FROM batch_dates
    )
    SELECT
        dosage_form,
        ROUND(AVG(end_date - compounding_date), 1) AS avg_lead_time_days,
        COUNT(*) AS batch_count
    FROM batch_lead_time
    WHERE compounding_date IS NOT NULL
    AND end_date IS NOT NULL
    GROUP BY dosage_form
    ORDER BY avg_lead_time_days DESC;
"""

df_lead_time_dosage_form = pd.read_sql_query(lead_time_dosage_form_query, engine)
df_lead_time_dosage_form

,dosage_form,avg_lead_time_days,batch_count
0,ENTERIC-COATED TABLET,17.0,2
1,EXTENDED RELEASE TABLET,14.2,5
2,FILM-COATED TABLET,13.0,422
3,MODIFIED RELEASE TABLET,13.0,2
4,BILAYER TABLET,11.5,17
5,SUSTAINED-RELEASE TABLET,10.5,265
6,TABLET,7.5,137
7,CAPSULE,4.9,174


In [8]:
fig = px.bar(
    df_lead_time_dosage_form,
    y='dosage_form',
    x='avg_lead_time_days',
    orientation='h',
    title='Average Lead Time by Dosage Form (2025)',
    labels={
        'avg_lead_time_days': 'Average Lead Time (Days)',
        'dosage_form': ''
    },
    text='avg_lead_time_days',
    color='avg_lead_time_days',
    color_continuous_scale='Blues'
)

fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='Average Lead Time (Days)')
)
fig.show()

## Q1 — Average Lead Time by Dosage Form
 
**Observation:**
Enteric-Coated Tablet has the longest average lead time at 17 days, followed by Extended Release Tablet (14.2 days), Modified Release Tablet (13 days), and Film-Coated Tablet (13 days). Bilayer Tablet sits at 11.5 days and Sustained-Release Tablet at 10.5 days. Plain Tablet (7.5 days) and Capsule (4.9 days) are the fastest to complete.
 
**Insight:**
The ranking follows a clear manufacturing logic — dosage forms requiring more processing stages naturally take longer. Capsules are fastest because they skip compression and coating entirely. Plain tablets skip coating. The specialized coating forms (enteric, extended release, modified release) take the longest because their coating processes are more technically demanding and time-sensitive than standard film coating. The 3.5x difference between the fastest (Capsule at 4.9 days) and slowest (Enteric-Coated Tablet at 17 days) reflects real formulation complexity, not scheduling inefficiency.
 
**Recommendation:**
Use these averages as dosage-form-specific lead time benchmarks rather than applying a single facility-wide target. A 21-day SLA is appropriate for enteric-coated and extended release products but unnecessarily generous for capsules and plain tablets. Consider tiered lead time targets — 7 days for capsules, 14 days for film-coated tablets, and 21 days for specialized coating forms — to create more meaningful performance monitoring.


## Q2 - What is the average lead time per month?

In [9]:
lead_time_monthly_query = """
    WITH batch_dates AS (
        SELECT
            f.jo_id,
            p.dosage_form,
            MAX(CASE WHEN f.stage = 'dry_blending' THEN d.full_date END) AS compounding_date,
            MAX(CASE WHEN f.stage = 'compression' THEN d.full_date END) AS compression_date,
            MAX(CASE WHEN f.stage = 'encapsulation' THEN d.full_date END) AS encapsulation_date,
            MAX(CASE WHEN f.stage = 'coating' THEN d.full_date END) AS coating_date
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        LEFT JOIN dim_date d ON f.date_id = d.date_id
        GROUP BY f.jo_id, p.dosage_form
    ),
    batch_lead_time AS (
        SELECT
            jo_id,
            dosage_form,
            compounding_date,
            CASE
                WHEN dosage_form = 'CAPSULE'
                    THEN encapsulation_date
                WHEN dosage_form IN ('FILM-COATED TABLET', 'SUSTAINED-RELEASE TABLET',
                                     'EXTENDED RELEASE TABLET', 'ENTERIC-COATED TABLET',
                                     'MODIFIED RELEASE TABLET')
                    THEN coating_date
                WHEN dosage_form IN ('TABLET', 'BILAYER TABLET')
                    THEN compression_date
                ELSE GREATEST(
                    COALESCE(compression_date, compounding_date),
                    COALESCE(encapsulation_date, compounding_date),
                    COALESCE(coating_date, compounding_date)
                )
            END AS end_date
        FROM batch_dates
    )
    SELECT
        TO_CHAR(compounding_date, 'YYYY-MM') AS month,
        TO_CHAR(compounding_date, 'Mon') AS month_short,
        ROUND(AVG(end_date - compounding_date), 1) AS avg_lead_time_days,
        COUNT(*) AS batch_count
    FROM batch_lead_time
    WHERE compounding_date IS NOT NULL
    AND end_date IS NOT NULL
    GROUP BY TO_CHAR(compounding_date, 'YYYY-MM'), TO_CHAR(compounding_date, 'Mon')
    ORDER BY month;
"""

df_lead_time_monthly = pd.read_sql_query(lead_time_monthly_query, engine)
df_lead_time_monthly

,month,month_short,avg_lead_time_days,batch_count
0,2025-01,Jan,12.2,93
1,2025-02,Feb,12.8,104
2,2025-03,Mar,10.1,56
3,2025-04,Apr,9.3,89
4,2025-05,May,8.1,85
5,2025-06,Jun,7.2,50
6,2025-07,Jul,10.0,94
7,2025-08,Aug,13.0,114
8,2025-09,Sep,12.0,120
9,2025-10,Oct,11.4,94


In [10]:
fig = px.line(
    df_lead_time_monthly,
    x='month_short',
    y='avg_lead_time_days',
    title='Average Lead Time by Month (2025)',
    labels={
        'avg_lead_time_days': 'Average Lead Time (Days)',
        'month_short': ''
    },
    text='avg_lead_time_days',
    markers=True
)

fig.update_traces(
    line=dict(color='steelblue', width=2),
    marker=dict(size=8, color='steelblue'),
    textposition='top center'
)

fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgrey', title='Average Lead Time (Days)')
)
fig.show()

## Q2 — Average Lead Time by Month
 
**Observation:**
Lead times peak in February (12.8 days) and August (13 days), then decline steadily through the second half of the year — dropping sharply from October (11.4 days) to November (5.2 days) and hitting the annual low in December (4.2 days). June is the mid-year low at 7.2 days before climbing again in July and August.
 
**Insight:**
The two peaks — February and August — align with your earlier production volume findings. February is a high-output month (108 batches) and August is part of the Q3 peak (116 batches). When the facility is processing more batches simultaneously, inter-stage waiting times increase, which extends lead times. The sharp November-December drop is likely driven by a shift toward simpler dosage forms (capsules, plain tablets) being prioritized to close out year-end production targets, rather than a genuine efficiency improvement.
 
**Recommendation:**
Investigate the product mix during November and December to confirm whether the lead time drop reflects dosage form composition rather than process improvement. If the Q3 peak consistently extends lead times, consider pre-building inventory of high-lead-time products (enteric-coated, extended release) in Q1 to reduce scheduling pressure during August and September.


## Q3 - Which generics have the longest average lead time?

In [11]:
lead_time_generic_query = """
    WITH batch_dates AS (
        SELECT
            f.jo_id,
            p.generic_name,
            p.dosage_form,
            MAX(CASE WHEN f.stage = 'dry_blending' THEN d.full_date END) AS compounding_date,
            MAX(CASE WHEN f.stage = 'compression' THEN d.full_date END) AS compression_date,
            MAX(CASE WHEN f.stage = 'encapsulation' THEN d.full_date END) AS encapsulation_date,
            MAX(CASE WHEN f.stage = 'coating' THEN d.full_date END) AS coating_date
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        LEFT JOIN dim_date d ON f.date_id = d.date_id
        GROUP BY f.jo_id, p.generic_name, p.dosage_form
    ),
    batch_lead_time AS (
        SELECT
            jo_id,
            generic_name,
            dosage_form,
            compounding_date,
            CASE
                WHEN dosage_form = 'CAPSULE'
                    THEN encapsulation_date
                WHEN dosage_form IN ('FILM-COATED TABLET', 'SUSTAINED-RELEASE TABLET',
                                     'EXTENDED RELEASE TABLET', 'ENTERIC-COATED TABLET',
                                     'MODIFIED RELEASE TABLET')
                    THEN coating_date
                WHEN dosage_form IN ('TABLET', 'BILAYER TABLET')
                    THEN compression_date
                ELSE GREATEST(
                    COALESCE(compression_date, compounding_date),
                    COALESCE(encapsulation_date, compounding_date),
                    COALESCE(coating_date, compounding_date)
                )
            END AS end_date
        FROM batch_dates
    )
    SELECT
        generic_name,
        ROUND(AVG(end_date - compounding_date), 1) AS avg_lead_time_days,
        COUNT(*) AS batch_count
    FROM batch_lead_time
    WHERE compounding_date IS NOT NULL
    AND end_date IS NOT NULL
    GROUP BY generic_name
    HAVING COUNT(*) >= 5
    ORDER BY avg_lead_time_days DESC
    LIMIT 15;
"""

df_lead_time_generic = pd.read_sql_query(lead_time_generic_query, engine)
df_lead_time_generic

,generic_name,avg_lead_time_days,batch_count
0,LEVETIRACETAM,27.1,8
1,TRANEXAMIC,19.0,7
2,ISOSORBIDE MONONITRATE,16.7,23
3,DAPAGLIFLOZIN,14.4,30
4,NAPROXEN SODIUM,13.8,13
5,SITAGLIPTIN (AS PHOSPHATE MONOHYDRATE) + METFO...,13.5,64
6,PARACETAMOL+PHENYLEPHRINE HCL+CHLORPHENIRAMINE,13.5,23
7,METFORMIN HYDROCHLORIDE,12.4,40
8,ATORVASTATIN CALCIUM,12.3,77
9,PARACETAMOL + THIAMINE MONONITRATE (VITAMIN B1...,12.0,49


In [12]:
# Truncate long generic names for readability
df_lead_time_generic['generic_short'] = df_lead_time_generic['generic_name'].apply(
    lambda x: x[:40] + '...' if len(x) > 40 else x
)

fig = px.bar(
    df_lead_time_generic,
    x='avg_lead_time_days',
    y='generic_short',
    orientation='h',
    title='Top 15 Generics by Average Lead Time (2025, min. 5 batches)',
    labels={
        'avg_lead_time_days': 'Average Lead Time (Days)',
        'generic_short': ''
    },
    text='avg_lead_time_days',
    color='avg_lead_time_days',
    color_continuous_scale='Blues'
)

fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='Average Lead Time (Days)')
)
fig.show()

## Q3 — Top 15 Generics by Average Lead Time
 
**Observation:**
Levetiracetam leads significantly at 27.1 days — the only generic exceeding the 21-day SLA by a meaningful margin. Tranexamic Acid follows at 19 days and Isosorbide Mononitrate at 16.7 days. The remaining 12 generics range from 10.8 to 14.4 days. All entries in this list required a minimum of 5 batches, making these statistically meaningful averages.
 
**Insight:**
Levetiracetam's 27.1-day average stands out as a clear outlier — it is 8 days above the 21-day SLA and nearly 8 days longer than the second-longest generic. This is unlikely to be a data artifact given the minimum 5-batch filter. Levetiracetam is an Extended Release Tablet, which already carries a 14.2-day dosage form average — suggesting this specific generic consistently takes significantly longer than other products in the same dosage form category. Tranexamic Acid and Isosorbide Mononitrate, while below the 21-day SLA, are worth monitoring as they are approaching the threshold.
 
**Recommendation:**
Conduct a batch-level deep dive on Levetiracetam to identify where the time is being lost — whether in the coating stage, inter-stage waiting, or scheduling gaps. Cross-reference with machine assignment to determine if coating machine selection is a contributing factor. For Tranexamic Acid and Isosorbide Mononitrate, flag them as watch-list products and track their lead times monthly to detect any upward trend before they breach the SLA.

In [17]:
lead_time_sla_query = """
    WITH batch_dates AS (
        SELECT
            f.jo_id,
            p.dosage_form,
            MAX(CASE WHEN f.stage = 'dry_blending' THEN d.full_date END) AS compounding_date,
            MAX(CASE WHEN f.stage = 'compression' THEN d.full_date END) AS compression_date,
            MAX(CASE WHEN f.stage = 'encapsulation' THEN d.full_date END) AS encapsulation_date,
            MAX(CASE WHEN f.stage = 'coating' THEN d.full_date END) AS coating_date
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        LEFT JOIN dim_date d ON f.date_id = d.date_id
        GROUP BY f.jo_id, p.dosage_form
    ),
    batch_lead_time AS (
        SELECT
            jo_id,
            dosage_form,
            compounding_date,
            CASE
                WHEN dosage_form = 'CAPSULE'
                    THEN encapsulation_date
                WHEN dosage_form IN ('FILM-COATED TABLET', 'SUSTAINED-RELEASE TABLET',
                                     'EXTENDED RELEASE TABLET', 'ENTERIC-COATED TABLET',
                                     'MODIFIED RELEASE TABLET')
                    THEN coating_date
                WHEN dosage_form IN ('TABLET', 'BILAYER TABLET')
                    THEN compression_date
                ELSE GREATEST(
                    COALESCE(compression_date, compounding_date),
                    COALESCE(encapsulation_date, compounding_date),
                    COALESCE(coating_date, compounding_date)
                )
            END AS end_date
        FROM batch_dates
    ),
    classified AS (
        SELECT
            dosage_form,
            (end_date - compounding_date) AS lead_time_days,
            CASE
                WHEN (end_date - compounding_date) <= 21 THEN 'Within 21 Days'
                ELSE 'Exceeded 21 Days'
            END AS sla_status
        FROM batch_lead_time
        WHERE compounding_date IS NOT NULL
        AND end_date IS NOT NULL
    )
    SELECT
        dosage_form,
        sla_status,
        COUNT(*) AS batch_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY dosage_form), 1) AS pct
    FROM classified
    GROUP BY dosage_form, sla_status
    ORDER BY dosage_form, sla_status;
"""

df_lead_time_sla = pd.read_sql_query(lead_time_sla_query, engine)
df_lead_time_sla

,dosage_form,sla_status,batch_count,pct
0,CAPSULE,Within 21 Days,174,100.0
1,FILM-COATED TABLET,Exceeded 21 Days,51,12.1
2,FILM-COATED TABLET,Within 21 Days,371,87.9
3,TABLET,Exceeded 21 Days,1,0.7
4,TABLET,Within 21 Days,136,99.3
5,SUSTAINED-RELEASE TABLET,Exceeded 21 Days,10,3.8
6,SUSTAINED-RELEASE TABLET,Within 21 Days,255,96.2
7,EXTENDED RELEASE TABLET,Within 21 Days,5,100.0
8,BILAYER TABLET,Exceeded 21 Days,2,11.8
9,BILAYER TABLET,Within 21 Days,15,88.2


In [19]:
fig = px.bar(
    df_lead_time_sla,
    x='dosage_form',
    y='pct',
    color='sla_status',
    title='Batch Completion Within 21-Day Target by Dosage Form (2025)',
    labels={
        'pct': '% of Batches',
        'dosage_form': '',
        'sla_status': ''
    },
    text='pct',
    color_discrete_map={
        'Within 21 Days': 'steelblue',
        'Exceeded 21 Days': 'lightcoral'
    },
    barmode='stack'
)

fig.update_traces(textposition='inside', texttemplate='%{text}%')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, tickangle=-30),
    yaxis=dict(showgrid=False, title='% of Batches', range=[0, 100]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

## Q4 — Batch Completion Within 21-Day Target by Dosage Form
 
**Observation:**
Capsule (100%), Tablet (99.3%), Extended Release Tablet (100%), Enteric-Coated Tablet (100%), and Modified Release Tablet (100%) all meet the 21-day SLA perfectly or near-perfectly. Sustained-Release Tablet achieves 96.2%. The two underperformers are Film-Coated Tablet at 87.9% (12.1% of batches exceeded 21 days) and Bilayer Tablet at 88.2% (11.8% exceeded).
 
**Insight:**
The 100% SLA compliance for Enteric-Coated and Extended Release Tablets is notable — despite having the longest average lead times in Q1, these dosage forms still consistently complete within 21 days. This confirms that the 21-day target is appropriately sized for complex coating forms. The underperformance of Film-Coated Tablets is the most operationally significant finding — it is the highest-volume dosage form in the facility (51.2% of total output) and yet 12.1% of its batches breach the SLA. Given the volume, this translates to a large absolute number of delayed batches. Bilayer Tablet's 11.8% breach rate is similarly concerning given the complexity of its two-layer compression process.
 
**Recommendation:**
Film-Coated Tablet SLA compliance should be the primary lead time improvement target for 2026. Investigate the specific batches that exceeded 21 days — identify whether the delays are concentrated in particular months (likely Q3), specific machines (cross-reference with KEVIN 48 utilization), or specific products (cross-reference with Q3 generic lead time list). For Bilayer Tablet, assess whether the two-layer compression process is creating scheduling bottlenecks that push coating dates beyond the SLA window.
 